# Generación de datos

Aquí se generan los datos utilizados por el e-commerce SkeletIA.

Los datos generados respetarán:

- Relaciones entre entidades
- Reglas de negocio
- Coherencia entre fechas, precios, costes, estados, pagos, valoraciones

El resultado se almacenará en `DataFrames` de pandas. Una vez generados los datos tendrán que validarse.  
Este será un paso previo a la carga en BigQuery.

In [1]:
# Imports

import random
import re

from datetime import datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP
from typing import Any

import pandas as pd

from faker import Faker
from faker.config import AVAILABLE_LOCALES

## Configuración general

Aquí se encuentran la configuración con las demandas del enunciado del ejercicio.

In [2]:
RANDOM_SEED = 42

NUM_CUSTOMERS = 500
NUM_PRODUCTS = 70
NUM_ORDERS = 2_000
NUM_ORDER_ITEMS = 4_500

REVIEW_PROBABILITY = 0.35

random.seed(RANDOM_SEED)
Faker.seed(RANDOM_SEED)

NOW = datetime.now().replace(microsecond=0)

## Generación de los datos para las tablas maestras

Primero se crean estos datos de referencia para que se puedan crear los datos de negocio con coherencia.

Para generar países y ciudades hago una función. Este e-commerce va a operar en 8 países y generaré 3 ciudades en cada país.  
De ahí obtendré los datos para generar los correspondientes DataFrame.

In [23]:
# Configuración para country y cities
COUNTRY_LOCALES = [
    "es_ES",
    "fr_FR",
    "de_DE",
    "it_IT",
    "pt_PT",
    "nl_NL",
    "fr_BE",
    "de_AT",
]

CITIES_PER_COUNTRY = 3

COUNTRY_CONFIG = {
    "es_ES": ("ES", "Spain"),
    "fr_FR": ("FR", "France"),
    "de_DE": ("DE", "Germany"),
    "it_IT": ("IT", "Italy"),
    "pt_PT": ("PT", "Portugal"),
    "nl_NL": ("NL", "Netherlands"),
    "fr_BE": ("BE", "Belgium"),
    "de_AT": ("AT", "Austria"),
}

In [24]:
def generate_geography(
    locales: list[str],
    cities_per_country: int = CITIES_PER_COUNTRY,
) -> tuple[list[dict], list[dict]]:
    """
    Genera países y ciudades utilizando Faker.

    Cada locale representa uno de los mercados en los que
    opera SkeletIA.
    """

    countries = []
    cities = []

    city_id = 1

    for country_id, locale in enumerate(locales, start=1):
        fake = Faker(locale)

        # Semilla específica por país para mantener
        # resultados reproducibles.
        fake.seed_instance(RANDOM_SEED + country_id)

        country_code, country_name = COUNTRY_CONFIG[locale]

        countries.append(
            {
                "country_id": country_id,
                "country_code": country_code,
                "country_name": country_name,
                "faker_locale": locale,
            }
        )

        generated_cities = set()

        while len(generated_cities) < cities_per_country:

            city_name = fake.city()

            if city_name in generated_cities:
                continue

            generated_cities.add(city_name)

            cities.append(
                {
                    "city_id": city_id,
                    "country_id": country_id,
                    "city_name": city_name,
                }
            )

            city_id += 1

    return countries, cities

In [25]:
COUNTRIES_DATA, CITIES_DATA = generate_geography(COUNTRY_LOCALES)

In [125]:
countries_df = pd.DataFrame(COUNTRIES_DATA)[["country_id", "country_code", "country_name"]]

cities_df = pd.DataFrame(CITIES_DATA)

In [27]:
countries_df

,country_id,country_code,country_name,faker_locale
0,1,ES,Spain,es_ES
1,2,FR,France,fr_FR
2,3,DE,Germany,de_DE
3,4,IT,Italy,it_IT
4,5,PT,Portugal,pt_PT
5,6,NL,Netherlands,nl_NL
6,7,BE,Belgium,fr_BE
7,8,AT,Austria,de_AT


In [28]:
cities_df

,city_id,country_id,city_name
0,1,1,Girona
1,2,1,Lleida
2,3,1,Sevilla
3,4,2,DijouxBourg
4,5,2,Gillet
5,6,2,Charpentier
6,7,3,Lübz
7,8,3,Grevenbroich
8,9,3,Hansestadttralsund
9,10,4,Lu Bagnu


Para los datos de las marcas sigo con el mismo proceso. Genero una función y mantengo el nombre de la tienda como marca propia.

In [9]:
def generate_brands(
    count: int = 10,
) -> list[dict]:
    """
    Genera marcas sintéticas manteniendo SkeletIA
    como marca propia con brand_id = 1.
    """

    fake = Faker("en_US")
    fake.seed_instance(RANDOM_SEED + 100)

    generated_names = set()

    while len(generated_names) < count - 1:

        brand_name = fake.company()[:80]

        if brand_name != "SkeletIA":
            generated_names.add(brand_name)

    brands = [
        {
            "brand_id": 1,
            "brand_name": "SkeletIA",
        }
    ]

    for brand_id, brand_name in enumerate(
        sorted(generated_names),
        start=2,
    ):
        brands.append(
            {
                "brand_id": brand_id,
                "brand_name": brand_name,
            }
        )

    return brands

In [10]:
BRANDS_DATA = generate_brands()

In [11]:
brands_df = pd.DataFrame(BRANDS_DATA)
brands_df

,brand_id,brand_name
0,1,SkeletIA
1,2,Adams Ltd
2,3,Alvarez Inc
3,4,Chan-Davidson
4,5,"Edwards, Murphy and Escobar"
5,6,"Grant, Oneill and Medina"
6,7,Johnson-Carter
7,8,Pope Group
8,9,Reed-Bass
9,10,"Steele, Bowman and Martin"


En el caso de las categorías o los canales por los que se llega al comercio creo más conveniente generarlos manualmente.  
La razón es que `Faker` lo haría ficticio y tiene que tener coherencia y relación con la lógica y características del negocio.

In [12]:
ACQUISITION_CHANNELS_DATA = [
    {
        "channel_id": 1,
        "channel_code": "organic",
        "channel_name": "Organic search",
    },
    {
        "channel_id": 2,
        "channel_code": "paid_ads",
        "channel_name": "Paid ads",
    },
    {
        "channel_id": 3,
        "channel_code": "social_media",
        "channel_name": "Social media",
    },
    {
        "channel_id": 4,
        "channel_code": "referral",
        "channel_name": "Referral",
    },
    {
        "channel_id": 5,
        "channel_code": "affiliate",
        "channel_name": "Affiliate",
    },
]


CATEGORIES_DATA = [
    {
        "category_id": 1,
        "category_name": "Smartphones",
        "description": "Smartphones and mobile devices",
    },
    {
        "category_id": 2,
        "category_name": "Laptops",
        "description": "Portable computers and ultrabooks",
    },
    {
        "category_id": 3,
        "category_name": "Tablets",
        "description": "Tablets and related devices",
    },
    {
        "category_id": 4,
        "category_name": "Audio",
        "description": "Headphones, speakers and audio accessories",
    },
    {
        "category_id": 5,
        "category_name": "Peripherals",
        "description": "Keyboards, mice, webcams and peripherals",
    },
    {
        "category_id": 6,
        "category_name": "Wearables",
        "description": "Smartwatches and fitness wearables",
    },
    {
        "category_id": 7,
        "category_name": "Smart Home",
        "description": "Connected home and IoT devices",
    },
    {
        "category_id": 8,
        "category_name": "Storage & Components",
        "description": "Storage devices and computer components",
    },
]

In [60]:
acquisition_channels_df = pd.DataFrame(ACQUISITION_CHANNELS_DATA)
categories_df = pd.DataFrame(CATEGORIES_DATA)

In [ ]:
acquisition_channels_df

,channel_id,channel_code,channel_name
0,1,organic,Organic search
1,2,paid_ads,Paid ads
2,3,social_media,Social media
3,4,referral,Referral
4,5,affiliate,Affiliate


In [15]:
categories_df

,category_id,category_name,description
0,1,Smartphones,Smartphones and mobile devices
1,2,Laptops,Portable computers and ultrabooks
2,3,Tablets,Tablets and related devices
3,4,Audio,"Headphones, speakers and audio accessories"
4,5,Peripherals,"Keyboards, mice, webcams and peripherals"
5,6,Wearables,Smartwatches and fitness wearables
6,7,Smart Home,Connected home and IoT devices
7,8,Storage & Components,Storage devices and computer components


## Generación de los datos de negocio

Para generar los datos de negocio voy a utilizar los ID que se han generado en las tablas maestras.  
Luego voy a necesitar un objeto `Faker` asociado a cada país.  
También auxiliares para generar precios, fechas, y distribuciones con coherencia.  

In [ ]:
# ID de country, city, channel, category, brand
COUNTRY_BY_ID = {country["country_id"]: country for country in COUNTRIES_DATA}

CITY_BY_ID = {city["city_id"]: city for city in CITIES_DATA}

CHANNEL_IDS = [channel["channel_id"] for channel in ACQUISITION_CHANNELS_DATA]

CATEGORY_IDS = [category["category_id"] for category in CATEGORIES_DATA]

BRAND_IDS = [brand["brand_id"] for brand in BRANDS_DATA]

BRAND_BY_ID = {brand["brand_id"]: brand for brand in BRANDS_DATA}

In [30]:
# Instancias de Faker para cada mercado
def safe_faker(locale: str) -> Faker:
    """
    Devuelve una instancia de Faker utilizando el locale solicitado.

    Si el locale no estuviera disponible, se utiliza en_US.
    """
    chosen_locale = (
        locale
        if locale in AVAILABLE_LOCALES
        else "en_US"
    )

    return Faker(chosen_locale)

In [31]:
FAKERS_BY_COUNTRY = {
    country["country_id"]: safe_faker(
        country["faker_locale"]
    )
    for country in COUNTRIES_DATA
}

In [19]:
# Auxiliares para precios, fechas y distribuciones
def money(value: float | Decimal) -> Decimal:
    """
    Convierte una cantidad monetaria en Decimal
    redondeado a dos decimales.
    """
    return Decimal(str(value)).quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP,
    )


def random_datetime(
    start: datetime,
    end: datetime,
) -> datetime:
    """
    Genera un datetime aleatorio entre start y end.
    """
    if end <= start:
        return start

    seconds = int((end - start).total_seconds())

    return start + timedelta(seconds=random.randint(0, seconds))


def weighted_choice(
    values: list[Any],
    weights: list[float],
) -> Any:
    """
    Selecciona un valor aplicando probabilidades diferentes.
    """
    return random.choices(
        values,
        weights=weights,
        k=1,
    )[0]

### Generación de datos de clientes

Se van a generar datos para 500 clientes.  
Hay que generar datos como nombre, apellido, teléfono, email, ciudad, canal, fecha...  
Faker se encarga de los datos personles y Python de las relaciones.

In [20]:
# Prefijos telefónicos reales de los países utilizados.
PHONE_PREFIXES = {
    "ES": "+34",
    "FR": "+33",
    "DE": "+49",
    "IT": "+39",
    "PT": "+351",
    "NL": "+31",
    "BE": "+32",
    "AT": "+43",
}


# Probabilidad aproximada de que un cliente
# llegue por cada canal de adquisición.
CHANNEL_WEIGHTS = {
    "organic": 35,
    "paid_ads": 25,
    "social_media": 20,
    "referral": 12,
    "affiliate": 8,
}

In [21]:
# Generación de clientes
def generate_customers(
    count: int = NUM_CUSTOMERS,
) -> list[dict[str, Any]]:
    """
    Genera clientes sintéticos relacionados con las
    ciudades y canales de adquisición existentes.
    """

    customers = []

    email_domains = [
        "gmail.com",
        "outlook.com",
        "proton.me",
        "icloud.com",
        "mail.com",
    ]

    for customer_id in range(1, count + 1):

        city = random.choice(CITIES_DATA)

        # a qué pais pertenece esa ciudad
        country = COUNTRY_BY_ID[city["country_id"]]
        fake = FAKERS_BY_COUNTRY[country["country_id"]]

        # nombre y apellido
        first_name = fake.first_name()
        last_name = fake.last_name()

        # email único
        email_user = re.sub(
            r"[^a-zA-Z0-9._-]",
            "",
            fake.user_name(),
        ).lower()

        email = (
            f"{email_user}.{customer_id:04d}"
            f"@{random.choice(email_domains)}"
        )

        # teléfono de contacto
        raw_phone = re.sub(
            r"[^0-9]",
            "",
            fake.phone_number(),
        )

        phone = (
            f"{PHONE_PREFIXES[country['country_code']]} "
            f"{raw_phone[-12:]}"
        )

        # canal de adquisición y su distribución
        channel_id = weighted_choice(
            [channel["channel_id"] for channel in ACQUISITION_CHANNELS_DATA],
            [CHANNEL_WEIGHTS[channel["channel_code"]] for channel in ACQUISITION_CHANNELS_DATA],
        )

        # fecha de registro
        registered_at = random_datetime(
            NOW - timedelta(days=900),
            NOW - timedelta(days=30),
        )

        # registro completo para cada cliente
        customers.append(
            {
                "customer_id": customer_id,
                "first_name": first_name[:80],
                "last_name": last_name[:120],
                "email": email[:180],
                "phone": phone[:30],
                "city_id": city["city_id"],
                "channel_id": channel_id,
                "registered_at": registered_at,
                "is_active": random.random() < 0.97,
            }
        )

    return customers

In [32]:
customers_rows = generate_customers()

print("Clientes generados:", len(customers_rows))

Clientes generados: 500


In [33]:
customers_df = pd.DataFrame(customers_rows)
customers_df.head()

,customer_id,first_name,last_name,email,phone,city_id,channel_id,registered_at,is_active
0,1,Isidro,Sanchez,nydia02.0001@proton.me,+34 34949542351,1,1,2024-10-20 06:35:58,True
1,2,Jannis,Kolm,kleinisabel.0002@mail.com,+43 350781618495,22,1,2026-01-01 22:38:03,True
2,3,Crescencia,Fernandez,arroyogisela.0003@outlook.com,+34 34877413164,3,1,2024-04-27 17:34:52,True
3,4,Milica,Rieder,michael55.0004@mail.com,+43 361928327648,23,2,2026-02-13 07:58:32,True
4,5,Isidoro,Morata,grandepaca.0005@outlook.com,+34 34745641395,1,3,2025-08-28 01:42:26,True


### Generación de datos de productos

Se van a generar 70 productos tecnológicos.  
Cada producto estará relacionado con una marca y categoría.  
El rango de precios dependerá de cada categoría.  
El coste será una proporción del PVP.

In [34]:
# Rangos de precios de las categorías
CATEGORY_PRICE_RANGES = {
    1: {"min_price": 249, "max_price": 1399},
    2: {"min_price": 499, "max_price": 2499},
    3: {"min_price": 199, "max_price": 1299},
    4: {"min_price": 29, "max_price": 599},
    5: {"min_price": 15, "max_price": 349},
    6: {"min_price": 49, "max_price": 699},
    7: {"min_price": 19, "max_price": 499},
    8: {"min_price": 39, "max_price": 999},
}

In [43]:
def generate_products(
    count: int = NUM_PRODUCTS,
) -> list[dict[str, Any]]:
    """
    Genera productos tecnológicos sintéticos.

    Cada producto:
    - pertenece a una categoría existente;
    - pertenece a una marca existente;
    - tiene un precio coherente con su categoría;
    - tiene un coste inferior al precio de venta;
    - tiene un nombre formado por marca + modelo;
    - tiene un SKU único.
    """

    products = []

    for product_id in range(1, count + 1):

        # Categoría y marca
        category_id = random.choice(CATEGORY_IDS)
        brand_id = random.choice(BRAND_IDS)

        price_range = CATEGORY_PRICE_RANGES[category_id]

        # precio actual de venta del producto
        current_sale_price = money(
            random.uniform(price_range["min_price"], price_range["max_price"])
        )

        # El coste estará aproximadamente entre
        # el 55 % y el 78 % del precio de venta.
        current_cost = money(current_sale_price * Decimal(str(random.uniform(0.55, 0.78))))

        # Nombre de la marca
        brand_name = BRAND_BY_ID[brand_id]["brand_name"]

        # Código de modelo de producto
        model_code = (
            f"{random.choice(['X', 'S', 'A', 'M', 'V'])}"
            f"{random.randint(100, 999)}"
        )

        # Nombre del producto
        product_name = (f"{brand_name} {model_code}")

        # Registro de producto
        products.append(
            {
                "product_id": product_id,
                "sku": f"SKL-{category_id:02d}-{product_id:04d}",
                "category_id": category_id,
                "brand_id": brand_id,
                "product_name": product_name,
                "current_sale_price": current_sale_price,
                "current_cost": current_cost,
                "stock": random.randint(0, 250),
                "is_active": random.random() < 0.94,
                "created_at": random_datetime(NOW - timedelta(days=1000), NOW),
            }
        )

    return products

In [44]:
products_rows = generate_products()

print("Productos generados:", len(products_rows))

Productos generados: 70


In [45]:
products_df = pd.DataFrame(products_rows)
products_df.head()

,product_id,sku,category_id,brand_id,product_name,current_sale_price,current_cost,stock,is_active,created_at
0,1,SKL-04-0001,4,2,Adams Ltd S857,75.76,44.36,99,True,2024-05-22 00:45:17
1,2,SKL-02-0002,2,1,SkeletIA A204,2384.92,1555.66,33,True,2025-10-08 00:19:46
2,3,SKL-08-0003,8,9,Reed-Bass X191,571.55,367.99,90,True,2026-06-22 19:12:32
3,4,SKL-06-0004,6,7,Johnson-Carter X842,57.02,36.79,230,True,2024-12-19 14:20:44
4,5,SKL-03-0005,3,7,Johnson-Carter A604,385.45,235.84,37,True,2025-10-13 18:09:16


### Generación de datos para orders

Se generan 2000 pedidos relacionados con clientes existentes.  
La fecha se genera teniendo en cuenta el registro de un cliente, no podrá haber hecho un pedido si todavía no era cliente.  
El envío no puede hacerse con fecha anterior a la que se ha realizado.  
La entrega tampoco podrá ser anterior al envío.  
Direcciones generadas con Faker.

In [46]:
def generate_orders(
    customers_rows: list[dict[str, Any]],
    count: int = NUM_ORDERS,
) -> list[dict[str, Any]]:
    """
    Genera pedidos sintéticos asociados a clientes existentes.

    Se respetan las siguientes relaciones temporales:

    registered_at <= order_date
    order_date <= shipped_at
    shipped_at <= delivered_at
    """

    orders = []

    # Margen de 10 días respecto a la fecha actual.
    end_date = NOW - timedelta(days=10)

    for order_id in range(1, count + 1):

        # Selección de cliente que realiza el pedido
        customer = random.choice(customers_rows)

        # Fecha desde la que puede hacer pedidos
        order_start = max(customer["registered_at"], NOW - timedelta(days=548))

        # Fecha de pedido
        order_date = random_datetime(order_start, end_date)

        # Estado del pedido (distribuido según [estado, %])
        status = weighted_choice(
            [
                "pending",
                "confirmed",
                "shipped",
                "delivered",
                "cancelled",
                "returned",
            ],
            [
                5,
                8,
                10,
                65,
                7,
                5,
            ],
        )

        # Fecha de envío y entrega
        shipped_at = None
        delivered_at = None

        # Con un pedido envíado se genera fecha de envío
        if status in {"shipped", "delivered", "returned"}:

            shipped_at = (order_date + timedelta(days=random.randint(1, 3)))

        # Con pedido entregado se genera fecha de entrega
        if status in {"delivered", "returned"}:

            delivered_at = (shipped_at + timedelta(days=random.randint(1, 5)))

        # Ciudad de envío (90% la misma de registro del cliente)
        if random.random() < 0.90:

            shipping_city_id = customer["city_id"]

        else:

            shipping_city_id = random.choice(CITIES_DATA)["city_id"]


        # País de la ciudad de envío
        shipping_city = CITY_BY_ID[shipping_city_id]

        shipping_country_id = shipping_city["country_id"]

        # Faker correspondiente al país
        shipping_fake = FAKERS_BY_COUNTRY[shipping_country_id]

        # Gastos de envío (distribuido según [gasto, %])
        shipping_cost = weighted_choice(
            [
                Decimal("0.00"),
                Decimal("4.99"),
                Decimal("7.99"),
                Decimal("9.99"),
            ],
            [
                35,
                30,
                23,
                12,
            ],
        )

        # Registro del pedido
        orders.append(
            {
                "order_id": order_id,
                "customer_id": customer["customer_id"],
                "status": status,
                "order_date": order_date,
                "shipped_at": shipped_at,
                "delivered_at": delivered_at,
                "shipping_recipient": (f"{customer['first_name']} {customer['last_name']}")[:200],
                "shipping_address_line1": (shipping_fake.street_address())[:200],
                "shipping_postal_code": (shipping_fake.postcode())[:20],
                "shipping_city_id": shipping_city_id,
                "shipping_cost": shipping_cost,
                "currency_code": "EUR",
            }
        )

    return orders

In [47]:
orders_rows = generate_orders(customers_rows)

print("Pedidos generados:", len(orders_rows))

Pedidos generados: 2000


In [48]:
orders_df = pd.DataFrame(orders_rows)
orders_df.head()

,order_id,customer_id,status,order_date,shipped_at,delivered_at,shipping_recipient,shipping_address_line1,shipping_postal_code,shipping_city_id,shipping_cost,currency_code
0,1,396,delivered,2025-08-03 07:11:11,2025-08-06 07:11:11,2025-08-08 07:11:11,Cornelius Neumeister,Igor-Kainz-Straße 647,5738,22,9.99,EUR
1,2,13,delivered,2025-09-01 11:38:28,2025-09-04 11:38:28,2025-09-05 11:38:28,Saskia Beckmann,Ronda de Gonzalo Baños 35 Apt. 71,26976,1,7.99,EUR
2,3,193,confirmed,2026-03-18 06:38:44,NaT,NaT,Susanna Prodi,"Strada Emilio, 6 Appartamento 5",13862,11,0.00,EUR
3,4,252,returned,2026-05-26 09:05:26,2026-05-27 09:05:26,2026-05-30 09:05:26,Bernard Mortier,613 Meunier Harbors,94020,20,9.99,EUR
4,5,460,delivered,2026-07-08 10:29:14,2026-07-10 10:29:14,2026-07-14 10:29:14,Marine Morel,Acceso de Alba Quiroga 58 Apt. 71,09007,3,4.99,EUR


### Generación de datos para order_items

Se generan aquí las 4500 líneas de pedido para los 2000 pedidos existentes.  
El precio y coste de cada producto son los que tenía en el momento de la compra, no dependen del precio almacenado en la tabla `products`.  
Para cada pedido se generan entre 1 y 7 productos.

In [49]:
def generate_order_items(
    orders_rows: list[dict[str, Any]],
    products_rows: list[dict[str, Any]],
    target_items: int = NUM_ORDER_ITEMS,
) -> list[dict[str, Any]]:
    """
    Genera las líneas de pedido.

    Cada pedido contiene entre 1 y 7 productos diferentes.

    La distribución favorece los pedidos pequeños para mantener
    una media aproximada de 2-3 productos por pedido y se ajusta
    posteriormente para generar exactamente target_items líneas.
    """

    order_count = len(orders_rows)

    minimum = order_count * 1
    maximum = order_count * 7

    if not minimum <= target_items <= maximum:
        raise ValueError("target_items debe permitir entre 1 y 7 líneas por pedido.")


    # Productos por pedido. Más frecuencia a menor cantidad de productos
    line_counts = random.choices(
        [1, 2, 3, 4, 5, 6, 7],
        weights=[
            30,
            40,
            18,
            7,
            3,
            1.5,
            0.5,
        ],
        k=order_count,
    )

    # Ajuste de la cantidad para tener las líneas indicadas en target_items
    difference = target_items - sum(line_counts)

    while difference != 0:

        # Faltan líneas.
        if difference > 0:

            candidates = [index for index, count in enumerate(line_counts) if count < 7]
            selected_index = random.choice(candidates)
            line_counts[selected_index] += 1
            difference -= 1

        # Sobran líneas.
        else:

            candidates = [index for index, count in enumerate(line_counts) if count > 1]
            selected_index = random.choice(candidates)
            line_counts[selected_index] -= 1
            difference += 1

    # Líneas para cada pedido
    order_items = []

    order_item_id = 1

    for order, line_count in zip(orders_rows, line_counts):

        # random.sample evita que el mismo producto
        # aparezca repetido dentro del pedido.
        selected_products = random.sample(products_rows, k=line_count)


        for product in selected_products:
            # Precio histórico
            price_factor = Decimal(str(random.uniform(0.85, 1.05)))
            unit_price = money(product["current_sale_price"] * price_factor)

            # Coste histórico
            historical_cost_factor = Decimal(str(random.uniform(0.90, 1.00)))
            historical_cost = product["current_cost"] * historical_cost_factor
            unit_cost = money(min(historical_cost, unit_price * Decimal("0.90")))

            # Unidades de producto (distribuido según [unidades, %])
            quantity = weighted_choice(
                [1, 2, 3],
                [80, 16, 4],
            )

            # Descuento (distribuido según [tipo de descuento, %])
            discount_percent = weighted_choice(
                [
                    Decimal("0.00"),
                    Decimal("5.00"),
                    Decimal("10.00"),
                    Decimal("15.00"),
                ],
                [
                    70,
                    15,
                    10,
                    5,
                ],
            )

            # Registro de la línea
            order_items.append(
                {
                    "order_item_id": order_item_id,
                    "order_id": order["order_id"],
                    "product_id": product["product_id"],
                    "quantity": quantity,
                    "unit_price": unit_price,
                    "unit_cost": unit_cost,
                    "discount_percent": discount_percent,
                }
            )

            order_item_id += 1

    # Comprobación de líneas
    if len(order_items) != target_items:
        raise RuntimeError("La generación no produjo exactamente {target_items} líneas.")

    return order_items

In [79]:
order_items_rows = generate_order_items(orders_rows, products_rows)

print("Líneas generadas:", len(order_items_rows))

Líneas generadas: 4500


In [51]:
order_items_df = pd.DataFrame(order_items_rows)
order_items_df.head()

,order_item_id,order_id,product_id,quantity,unit_price,unit_cost,discount_percent
0,1,1,47,2,44.25,27.57,10.00
1,2,1,35,1,421.21,260.41,5.00
2,3,2,59,1,172.12,107.07,0.00
3,4,3,7,1,605.68,379.66,0.00
4,5,4,61,1,1622.65,893.67,5.00


### Generación de datos para payments

Cada pedido tendrá un único pago.  
El importe se calcula a través del pedido, teniendo en cuenta cantidades, precios, gastos de envío y descuentos.  
Se genera un estado de pago coherente con el estado del envío.

In [52]:
def generate_payments(
    orders_rows: list[dict[str, Any]],
    order_items_rows: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    Genera un pago por cada pedido.

    El importe se calcula como:

    suma de líneas con descuento + gastos de envío.
    """

    # Líneas por pedido
    items_by_order = {}

    for item in order_items_rows:

        items_by_order.setdefault(item["order_id"], []).append(item)

    payments = []

    # Recorrer todos los pedidos, por cada pedido se calculan los productos
    for payment_id, order in enumerate(orders_rows, start=1):

        order_total = Decimal("0.00")

        for item in items_by_order[order["order_id"]]:

            discount_factor = Decimal("1.00") - (item["discount_percent"] / Decimal("100"))
            line_total = Decimal(item["quantity"]) * item["unit_price"] * discount_factor
            order_total += line_total

        # Gastos de envío
        order_total += order["shipping_cost"]

        order_total = money(order_total)

        # Método de pago (distribuido según [método, %])
        payment_method = weighted_choice(
            [
                "card",
                "paypal",
                "bank_transfer",
                "apple_pay",
                "google_pay",
            ],
            [
                70,
                15,
                5,
                5,
                5,
            ],
        )

        # Estado de pago según el estado del pedido
        if order["status"] == "pending":
            payment_status = "pending"

        elif order["status"] == "returned":
            payment_status = "refunded"

        elif order["status"] == "cancelled":
            payment_status = weighted_choice(
                [
                    "failed",
                    "refunded",
                ],
                [
                    40,
                    60,
                ],
            )

        else:
            payment_status = "completed"

        # Fecha de pago
        payment_date = order["order_date"] + timedelta(days=random.randint(0, 1))

        # Última actualización del estado
        if order["status"] == "returned":
            status_updated_at = min(NOW, order["delivered_at"] + timedelta(days=random.randint(1, 10)))

        elif order["status"] == "cancelled":
            status_updated_at = (order["order_date"] + timedelta(days=1))

        else:
            status_updated_at = payment_date


        # Nunca puede actualizarse antes de haberse creado el pago.
        status_updated_at = max(status_updated_at, payment_date)

        # Registro del pago
        payments.append(
            {
                "payment_id": payment_id,
                "order_id": order["order_id"],
                "payment_method": payment_method,
                "status": payment_status,
                "amount": order_total,
                "payment_date": payment_date,
                "status_updated_at": status_updated_at,
                "external_reference": f"PAY-{order['order_id']:08d}",
            }
        )

    return payments

In [80]:
payments_rows = generate_payments(orders_rows, order_items_rows)

print("Pagos generados:", len(payments_rows))

Pagos generados: 2000


In [54]:
payments_df = pd.DataFrame(payments_rows)
payments_df.head()

,payment_id,order_id,payment_method,status,amount,payment_date,status_updated_at,external_reference
0,1,1,card,completed,489.79,2025-08-03 07:11:11,2025-08-03 07:11:11,PAY-00000001
1,2,2,card,completed,180.11,2025-09-01 11:38:28,2025-09-01 11:38:28,PAY-00000002
2,3,3,card,completed,605.68,2026-03-18 06:38:44,2026-03-18 06:38:44,PAY-00000003
3,4,4,card,refunded,2168.00,2026-05-26 09:05:26,2026-06-01 09:05:26,PAY-00000004
4,5,5,card,completed,1173.26,2026-07-08 10:29:14,2026-07-08 10:29:14,PAY-00000005


### Generación de datos para reviews

Únicamente productos entregados pueden tener valoración.  
El 35% de los entregados recibe valoración, una única valoración por pedido.  
La fecha siempre va a ser posterior a la fecha de entrega.

In [55]:
# Posibles comentarios en diferentes idiomas
REVIEW_TEXTS = {
    "es": {
        1: [
            "El producto no cumplió mis expectativas.",
            "No volvería a comprar este modelo.",
            "La experiencia general ha sido decepcionante.",
            "La calidad está bastante por debajo de lo esperado.",
            "No estoy satisfecho con la compra.",
        ],
        2: [
            "Funciona, pero esperaba bastante más.",
            "El producto es aceptable, aunque tiene varios aspectos mejorables.",
            "Cumple lo básico, pero no destaca especialmente.",
            "La relación calidad-precio podría ser mejor.",
            "No está mal, pero no ha terminado de convencerme.",
        ],
        3: [
            "El producto cumple correctamente con su función.",
            "Una compra razonable por el precio.",
            "La experiencia ha sido correcta, sin grandes sorpresas.",
            "Funciona como esperaba y ofrece un rendimiento aceptable.",
            "Un producto equilibrado para un uso normal.",
        ],
        4: [
            "Muy buen producto, estoy satisfecho con la compra.",
            "Buen rendimiento y una experiencia general positiva.",
            "El producto ha cumplido muy bien mis expectativas.",
            "Buena calidad y funcionamiento estable.",
            "Lo volvería a comprar sin demasiadas dudas.",
        ],
        5: [
            "Excelente producto, estoy muy satisfecho.",
            "Ha superado mis expectativas.",
            "Muy buena calidad y un rendimiento excelente.",
            "Una compra totalmente recomendable.",
            "Muy satisfecho con el producto y con su funcionamiento.",
        ],
    },


    "fr": {
        1: [
            "Le produit n'a pas répondu à mes attentes.",
            "Je n'achèterais pas ce modèle à nouveau.",
            "L'expérience générale a été décevante.",
            "La qualité est bien inférieure à ce que j'attendais.",
            "Je ne suis pas satisfait de cet achat.",
        ],
        2: [
            "Le produit fonctionne, mais je m'attendais à mieux.",
            "Le produit est acceptable, mais plusieurs aspects pourraient être améliorés.",
            "Il remplit les fonctions de base, sans plus.",
            "Le rapport qualité-prix pourrait être meilleur.",
            "Ce n'est pas mauvais, mais le produit ne m'a pas vraiment convaincu.",
        ],
        3: [
            "Le produit remplit correctement sa fonction.",
            "Un achat raisonnable pour ce prix.",
            "L'expérience est correcte, sans grande surprise.",
            "Le produit fonctionne comme prévu avec des performances acceptables.",
            "Un produit équilibré pour une utilisation normale.",
        ],
        4: [
            "Très bon produit, je suis satisfait de mon achat.",
            "Bonnes performances et expérience globalement positive.",
            "Le produit a très bien répondu à mes attentes.",
            "Bonne qualité et fonctionnement stable.",
            "Je pourrais acheter ce produit à nouveau.",
        ],
        5: [
            "Excellent produit, j'en suis très satisfait.",
            "Le produit a dépassé mes attentes.",
            "Très bonne qualité et excellentes performances.",
            "Un achat que je recommande sans hésiter.",
            "Très satisfait du produit et de son fonctionnement.",
        ],
    },


    "de": {
        1: [
            "Das Produkt hat meine Erwartungen nicht erfüllt.",
            "Ich würde dieses Modell nicht noch einmal kaufen.",
            "Die Gesamterfahrung war enttäuschend.",
            "Die Qualität liegt deutlich unter meinen Erwartungen.",
            "Ich bin mit diesem Kauf nicht zufrieden.",
        ],
        2: [
            "Das Produkt funktioniert, aber ich hatte mehr erwartet.",
            "Das Produkt ist akzeptabel, könnte aber in mehreren Punkten besser sein.",
            "Die grundlegenden Funktionen werden erfüllt, mehr aber auch nicht.",
            "Das Preis-Leistungs-Verhältnis könnte besser sein.",
            "Nicht schlecht, aber das Produkt hat mich nicht vollständig überzeugt.",
        ],
        3: [
            "Das Produkt erfüllt seinen Zweck zuverlässig.",
            "Für diesen Preis ist es ein vernünftiger Kauf.",
            "Die Erfahrung war insgesamt in Ordnung.",
            "Das Produkt funktioniert wie erwartet und bietet eine akzeptable Leistung.",
            "Ein ausgewogenes Produkt für den normalen Gebrauch.",
        ],
        4: [
            "Sehr gutes Produkt, ich bin mit dem Kauf zufrieden.",
            "Gute Leistung und insgesamt eine positive Erfahrung.",
            "Das Produkt hat meine Erwartungen sehr gut erfüllt.",
            "Gute Qualität und zuverlässige Funktion.",
            "Ich würde das Produkt wahrscheinlich erneut kaufen.",
        ],
        5: [
            "Ausgezeichnetes Produkt, ich bin sehr zufrieden.",
            "Das Produkt hat meine Erwartungen übertroffen.",
            "Sehr gute Qualität und hervorragende Leistung.",
            "Ein absolut empfehlenswerter Kauf.",
            "Ich bin mit dem Produkt und seiner Leistung sehr zufrieden.",
        ],
    },


    "it": {
        1: [
            "Il prodotto non ha soddisfatto le mie aspettative.",
            "Non comprerei di nuovo questo modello.",
            "L'esperienza complessiva è stata deludente.",
            "La qualità è decisamente inferiore alle aspettative.",
            "Non sono soddisfatto dell'acquisto.",
        ],
        2: [
            "Il prodotto funziona, ma mi aspettavo di più.",
            "È accettabile, anche se ci sono diversi aspetti da migliorare.",
            "Svolge le funzioni di base, ma senza distinguersi.",
            "Il rapporto qualità-prezzo potrebbe essere migliore.",
            "Non è male, ma non mi ha convinto del tutto.",
        ],
        3: [
            "Il prodotto svolge correttamente la sua funzione.",
            "Un acquisto ragionevole considerando il prezzo.",
            "L'esperienza è stata nella media, senza particolari sorprese.",
            "Funziona come previsto e offre prestazioni accettabili.",
            "Un prodotto equilibrato per un utilizzo normale.",
        ],
        4: [
            "Ottimo prodotto, sono soddisfatto dell'acquisto.",
            "Buone prestazioni e un'esperienza complessivamente positiva.",
            "Il prodotto ha soddisfatto molto bene le mie aspettative.",
            "Buona qualità e funzionamento affidabile.",
            "Lo comprerei probabilmente di nuovo.",
        ],
        5: [
            "Prodotto eccellente, sono molto soddisfatto.",
            "Ha superato le mie aspettative.",
            "Qualità molto buona e prestazioni eccellenti.",
            "Un acquisto che consiglio senza esitazioni.",
            "Sono molto soddisfatto del prodotto e delle sue prestazioni.",
        ],
    },


    "pt": {
        1: [
            "O produto não correspondeu às minhas expectativas.",
            "Não voltaria a comprar este modelo.",
            "A experiência geral foi dececionante.",
            "A qualidade ficou bastante abaixo do esperado.",
            "Não fiquei satisfeito com esta compra.",
        ],
        2: [
            "O produto funciona, mas esperava bastante mais.",
            "É aceitável, embora existam vários aspetos a melhorar.",
            "Cumpre as funções básicas, mas pouco mais.",
            "A relação qualidade-preço poderia ser melhor.",
            "Não é mau, mas não me convenceu completamente.",
        ],
        3: [
            "O produto cumpre corretamente a sua função.",
            "Uma compra razoável tendo em conta o preço.",
            "A experiência foi normal, sem grandes surpresas.",
            "Funciona como esperado e tem um desempenho aceitável.",
            "Um produto equilibrado para uma utilização normal.",
        ],
        4: [
            "Muito bom produto, fiquei satisfeito com a compra.",
            "Bom desempenho e uma experiência global positiva.",
            "O produto correspondeu muito bem às minhas expectativas.",
            "Boa qualidade e funcionamento estável.",
            "Provavelmente voltaria a comprar este produto.",
        ],
        5: [
            "Excelente produto, estou muito satisfeito.",
            "Superou as minhas expectativas.",
            "Muito boa qualidade e excelente desempenho.",
            "Uma compra que recomendo sem hesitação.",
            "Estou muito satisfeito com o produto e o seu funcionamento.",
        ],
    },


    "nl": {
        1: [
            "Het product voldeed niet aan mijn verwachtingen.",
            "Ik zou dit model niet opnieuw kopen.",
            "De algemene ervaring was teleurstellend.",
            "De kwaliteit ligt duidelijk onder mijn verwachtingen.",
            "Ik ben niet tevreden met deze aankoop.",
        ],
        2: [
            "Het product werkt, maar ik had meer verwacht.",
            "Het product is acceptabel, maar er zijn verschillende verbeterpunten.",
            "Het voldoet aan de basisverwachtingen, maar niet veel meer.",
            "De prijs-kwaliteitverhouding zou beter kunnen.",
            "Niet slecht, maar het product heeft mij niet volledig overtuigd.",
        ],
        3: [
            "Het product doet wat het moet doen.",
            "Een redelijke aankoop voor deze prijs.",
            "De ervaring was gemiddeld, zonder grote verrassingen.",
            "Het werkt zoals verwacht en levert acceptabele prestaties.",
            "Een evenwichtig product voor normaal gebruik.",
        ],
        4: [
            "Zeer goed product, ik ben tevreden met mijn aankoop.",
            "Goede prestaties en over het algemeen een positieve ervaring.",
            "Het product voldeed zeer goed aan mijn verwachtingen.",
            "Goede kwaliteit en betrouwbare werking.",
            "Ik zou dit product waarschijnlijk opnieuw kopen.",
        ],
        5: [
            "Uitstekend product, ik ben zeer tevreden.",
            "Het product heeft mijn verwachtingen overtroffen.",
            "Zeer goede kwaliteit en uitstekende prestaties.",
            "Een aankoop die ik zeker zou aanbevelen.",
            "Ik ben zeer tevreden met het product en de prestaties.",
        ],
    },


    "en": {
        1: [
            "The product did not meet my expectations.",
            "I would not buy this model again.",
            "The overall experience was disappointing.",
            "The quality was below what I expected.",
            "I am not satisfied with this purchase.",
        ],
        2: [
            "It works, but I expected more.",
            "The product is acceptable, although several aspects could be improved.",
            "It covers the basic functions but does not stand out.",
            "The value for money could be better.",
            "It is not bad, but it did not fully convince me.",
        ],
        3: [
            "The product does its job correctly.",
            "A reasonable purchase for the price.",
            "The overall experience was average.",
            "It works as expected and provides acceptable performance.",
            "A balanced product for normal use.",
        ],
        4: [
            "Very good product. I am satisfied with the purchase.",
            "Good performance and a positive overall experience.",
            "The product met my expectations very well.",
            "Good quality and reliable performance.",
            "I would probably buy this product again.",
        ],
        5: [
            "Excellent product. I am very satisfied.",
            "The product exceeded my expectations.",
            "Very good quality and excellent performance.",
            "A purchase I would definitely recommend.",
            "I am very satisfied with the product and its performance.",
        ],
    },
}

In [56]:
def generate_reviews(
    orders_rows: list[dict[str, Any]],
    order_items_rows: list[dict[str, Any]],
    customers_rows: list[dict[str, Any]],
    probability: float = REVIEW_PROBABILITY,
) -> list[dict[str, Any]]:
    """
    Genera valoraciones para aproximadamente el 35 %
    de las líneas pertenecientes a pedidos entregados.

    El idioma de cada comentario se determina a partir
    del país del cliente que realizó el pedido.

    Cada order_item puede tener como máximo una review.
    """
    
    # Localizar pedido y cliente
    order_by_id = {order["order_id"]: order for order in orders_rows}
    customer_by_id = {customer["customer_id"]: customer for customer in customers_rows}

    reviews = []
    review_id = 1


    for item in order_items_rows:

        order = order_by_id[item["order_id"]]

        # Si no se ha entregado, descarta
        if order["status"] != "delivered":
            continue

        # Si no descartas únicamente el 35% aprox tendrá valoración
        if random.random() >= probability:
            continue

        # Cliente del pedido
        customer = customer_by_id[order["customer_id"]]

        city = CITY_BY_ID[customer["city_id"]]

        country = COUNTRY_BY_ID[city["country_id"]]

        language = country["faker_locale"].split("_")[0]

        # Si el idioma no tiene reviews se usa inglés
        if language not in REVIEW_TEXTS:
            language = "en"

        # Puntuación (distribuida en [puntuación, %])
        rating = weighted_choice(
            [1, 2, 3, 4, 5],
            [4, 6, 15, 35, 40],
        )

        # Fecha de entrega
        delivered_at = order["delivered_at"]

        # Fecha de review
        review_date = min(NOW, delivered_at + timedelta(days=random.randint(1, 30)))

        # Comentario en idioma de cliente y coherente con rating
        comment = random.choice(REVIEW_TEXTS[language][rating])

        # Registro de review
        reviews.append(
            {
                "review_id": review_id,
                "order_item_id": item["order_item_id"],
                "rating": rating,
                "comment": comment,
                "review_date": review_date,
            }
        )

        review_id += 1


    return reviews

In [81]:
reviews_rows = generate_reviews(orders_rows, order_items_rows, customers_rows)

print("Reviews generadas:", len(reviews_rows))

Reviews generadas: 988


In [59]:
reviews_df = pd.DataFrame(reviews_rows)
reviews_df.head()

,review_id,order_item_id,rating,comment,review_date
0,1,1,4,Das Produkt hat meine Erwartungen sehr gut erf...,2025-08-22 07:11:11
1,2,2,5,Ich bin mit dem Produkt und seiner Leistung se...,2025-08-09 07:11:11
2,3,3,5,Das Produkt hat meine Erwartungen übertroffen.,2025-10-04 11:38:28
3,4,8,1,La qualité est bien inférieure à ce que j'atte...,2026-08-07 10:29:14
4,5,9,5,Le produit a dépassé mes attentes.,2026-07-22 10:29:14


Por comodidad agrupo todos los `DataFrame` en un diccionario

In [126]:
DATAFRAMES = {
    "countries": countries_df,
    "cities": cities_df,
    "acquisition_channels": acquisition_channels_df,
    "categories": categories_df,
    "brands": brands_df,
    "customers": customers_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "reviews": reviews_df,
}

## Validación global de datos antes de inserción a BigQuery

Se comprobará la integridad del conjunto: cantidades, claves, unicidad, rangos, fechas, reglas...

In [105]:
validation_results = []


def check(description: str, condition: bool) -> None:
    """
    Registra y muestra el resultado de una comprobación.
    """

    status = "OK" if condition else "ERROR"

    validation_results.append(
        {
            "check": description,
            "status": status,
        }
    )

    print(
        f"[{status}] {description}"
    )

In [106]:
print("\n=== 1. CANTIDADES ===\n")

check("500 clientes", len(customers_df) == NUM_CUSTOMERS)

check("70 productos", len(products_df) == NUM_PRODUCTS)

check("2000 pedidos", len(orders_df) == NUM_ORDERS)

check("4500 líneas de pedido", len(order_items_df) == NUM_ORDER_ITEMS)

check("Un pago por pedido", len(payments_df) == NUM_ORDERS)


=== 1. CANTIDADES ===

[OK] 500 clientes
[OK] 70 productos
[OK] 2000 pedidos
[OK] 4500 líneas de pedido
[OK] Un pago por pedido


In [107]:
delivered_order_ids = set(orders_df.loc[orders_df["status"] == "delivered", "order_id"])

delivered_items = order_items_df[order_items_df["order_id"].isin(delivered_order_ids)]

review_percentage = (len(reviews_df) / len(delivered_items) * 100)

check(
    "Reviews aproximadamente sobre el 35 % de productos entregados",
    30 <= review_percentage <= 40
)

print(f"Porcentaje real de reviews: {review_percentage:.2f}%")

[OK] Reviews aproximadamente sobre el 35 % de productos entregados
Porcentaje real de reviews: 35.43%


In [108]:
print("\n=== 2. CLAVES PRIMARIAS ===\n")

PRIMARY_KEYS = {
    "countries": "country_id",
    "cities": "city_id",
    "acquisition_channels": "channel_id",
    "categories": "category_id",
    "brands": "brand_id",
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id",
    "reviews": "review_id",
}


for table_name, primary_key in PRIMARY_KEYS.items():

    df = DATAFRAMES[table_name]

    check(f"{table_name}.{primary_key} sin duplicados", df[primary_key].is_unique)

    check(f"{table_name}.{primary_key} sin NULL", df[primary_key].notna().all())


=== 2. CLAVES PRIMARIAS ===

[OK] countries.country_id sin duplicados
[OK] countries.country_id sin NULL
[OK] cities.city_id sin duplicados
[OK] cities.city_id sin NULL
[OK] acquisition_channels.channel_id sin duplicados
[OK] acquisition_channels.channel_id sin NULL
[OK] categories.category_id sin duplicados
[OK] categories.category_id sin NULL
[OK] brands.brand_id sin duplicados
[OK] brands.brand_id sin NULL
[OK] customers.customer_id sin duplicados
[OK] customers.customer_id sin NULL
[OK] products.product_id sin duplicados
[OK] products.product_id sin NULL
[OK] orders.order_id sin duplicados
[OK] orders.order_id sin NULL
[OK] order_items.order_item_id sin duplicados
[OK] order_items.order_item_id sin NULL
[OK] payments.payment_id sin duplicados
[OK] payments.payment_id sin NULL
[OK] reviews.review_id sin duplicados
[OK] reviews.review_id sin NULL


In [109]:
print("\n=== 3. CLAVES FORÁNEAS ===\n")


def valid_fk(
    child_df: pd.DataFrame,
    child_column: str,
    parent_df: pd.DataFrame,
    parent_column: str,
) -> bool:
    """
    Comprueba que todos los valores de una FK
    existan en la tabla padre.
    """

    return child_df[child_column].isin(parent_df[parent_column]).all()

check(
    "cities.country_id -> countries.country_id",
    valid_fk(
        cities_df,
        "country_id",
        countries_df,
        "country_id",
    )
)

check(
    "customers.city_id -> cities.city_id",
    valid_fk(
        customers_df,
        "city_id",
        cities_df,
        "city_id",
    )
)

check(
    "customers.channel_id -> acquisition_channels.channel_id",
    valid_fk(
        customers_df,
        "channel_id",
        acquisition_channels_df,
        "channel_id",
    )
)

check(
    "products.category_id -> categories.category_id",
    valid_fk(
        products_df,
        "category_id",
        categories_df,
        "category_id",
    )
)

check(
    "products.brand_id -> brands.brand_id",
    valid_fk(
        products_df,
        "brand_id",
        brands_df,
        "brand_id",
    )
)

check(
    "orders.customer_id -> customers.customer_id",
    valid_fk(
        orders_df,
        "customer_id",
        customers_df,
        "customer_id",
    )
)

check(
    "orders.shipping_city_id -> cities.city_id",
    valid_fk(
        orders_df,
        "shipping_city_id",
        cities_df,
        "city_id",
    )
)

check(
    "order_items.order_id -> orders.order_id",
    valid_fk(
        order_items_df,
        "order_id",
        orders_df,
        "order_id",
    )
)

check(
    "order_items.product_id -> products.product_id",
    valid_fk(
        order_items_df,
        "product_id",
        products_df,
        "product_id",
    )
)

check(
    "payments.order_id -> orders.order_id",
    valid_fk(
        payments_df,
        "order_id",
        orders_df,
        "order_id",
    )
)

check(
    "reviews.order_item_id -> order_items.order_item_id",
    valid_fk(
        reviews_df,
        "order_item_id",
        order_items_df,
        "order_item_id",
    )
)


=== 3. CLAVES FORÁNEAS ===

[OK] cities.country_id -> countries.country_id
[OK] customers.city_id -> cities.city_id
[OK] customers.channel_id -> acquisition_channels.channel_id
[OK] products.category_id -> categories.category_id
[OK] products.brand_id -> brands.brand_id
[OK] orders.customer_id -> customers.customer_id
[OK] orders.shipping_city_id -> cities.city_id
[OK] order_items.order_id -> orders.order_id
[OK] order_items.product_id -> products.product_id
[OK] payments.order_id -> orders.order_id
[OK] reviews.order_item_id -> order_items.order_item_id


In [110]:
print("\n=== 4. UNICIDAD ===\n")

check(
    "country_code únicos",
    countries_df["country_code"].is_unique
)

check(
    "country_name únicos",
    countries_df["country_name"].is_unique
)

check(
    "Email de clientes únicos",
    customers_df["email"].is_unique
)

check(
    "SKU de productos únicos",
    products_df["sku"].is_unique
)

check(
    "Referencias externas de pagos únicas",
    payments_df["external_reference"].is_unique
)

check(
    "Sin ciudades duplicadas dentro de un país",
    not cities_df.duplicated(
        subset=["country_id", "city_name"]).any()
)

check(
    "Sin productos repetidos dentro del mismo pedido",
    not order_items_df.duplicated(
        subset=["order_id", "product_id"]).any()
)

check(
    "Máximo una review por línea de pedido",
    reviews_df["order_item_id"].is_unique
)

check(
    "Exactamente un pago por pedido",
    payments_df["order_id"].is_unique
)


=== 4. UNICIDAD ===

[OK] country_code únicos
[OK] country_name únicos
[OK] Email de clientes únicos
[OK] SKU de productos únicos
[OK] Referencias externas de pagos únicas
[OK] Sin ciudades duplicadas dentro de un país
[OK] Sin productos repetidos dentro del mismo pedido
[OK] Máximo una review por línea de pedido
[OK] Exactamente un pago por pedido


In [111]:
print("\n=== 5. RANGOS Y DOMINIOS ===\n")

check(
    "Precios de producto no negativos",
    (products_df["current_sale_price"] >= 0).all()
)

check(
    "Costes de producto no negativos",
    (products_df["current_cost"] >= 0).all()
)

check(
    "Coste de producto inferior al precio",
    (products_df["current_cost"] < products_df["current_sale_price"]).all()
)

check(
    "Stock no negativo",
    (products_df["stock"] >= 0).all()
)

check(
    "Cantidad de productos mayor que 0",
    (order_items_df["quantity"] > 0).all()
)

check(
    "unit_price no negativo",
    (order_items_df["unit_price"] >= 0).all()
)

check(
    "unit_cost no negativo",
    (order_items_df["unit_cost"] >= 0).all()
)

check(
    "unit_cost inferior a unit_price",
    (order_items_df["unit_cost"] < order_items_df["unit_price"]).all()
)

check(
    "Descuentos entre 0 y 100 %",
    order_items_df["discount_percent"].between(0, 100).all()
)

items_per_order = order_items_df.groupby("order_id").size()

check(
    "Todos los pedidos tienen entre 1 y 7 productos",
    items_per_order.between(1, 7).all()
)

check(
    "Los 2000 pedidos tienen al menos una línea",
    len(items_per_order) == NUM_ORDERS
)


=== 5. RANGOS Y DOMINIOS ===

[OK] Precios de producto no negativos
[OK] Costes de producto no negativos
[OK] Coste de producto inferior al precio
[OK] Stock no negativo
[OK] Cantidad de productos mayor que 0
[OK] unit_price no negativo
[OK] unit_cost no negativo
[OK] unit_cost inferior a unit_price
[OK] Descuentos entre 0 y 100 %
[OK] Todos los pedidos tienen entre 1 y 7 productos
[OK] Los 2000 pedidos tienen al menos una línea


In [112]:
VALID_ORDER_STATUSES = {
    "cancelled",
    "confirmed",
    "delivered",
    "pending",
    "returned",
    "shipped",    
}

VALID_PAYMENT_METHODS = {
    "apple_pay",
    "bank_transfer",
    "card",
    "cash_on_delivery",
    "google_pay",
    "paypal",
    "samsung_pay",
}

VALID_PAYMENT_STATUSES = {
    "completed",
    "failed",
    "pending",
    "refunded",
}

check(
    "Estados de pedido válidos",
    orders_df["status"].isin(VALID_ORDER_STATUSES).all()
)

check(
    "Estados de pago válidos",
    payments_df["status"].isin(VALID_PAYMENT_STATUSES).all()
)

check(
    "Métodos de pago válidos",
    payments_df["payment_method"].isin(VALID_PAYMENT_METHODS).all()
)

check(
    "Ratings entre 1 y 5",
    reviews_df["rating"].between(1, 5).all()
)

[OK] Estados de pedido válidos
[OK] Estados de pago válidos
[OK] Métodos de pago válidos
[OK] Ratings entre 1 y 5


In [113]:
print("\n=== 6. COHERENCIA TEMPORAL ===\n")

orders_with_customers = orders_df.merge(
    customers_df[["customer_id","registered_at"]],
    on="customer_id",
    how="left"
)

check(
    "Ningún pedido anterior al registro del cliente",
    (orders_with_customers["order_date"] >= orders_with_customers["registered_at"]).all()
)

shipped_orders = orders_df[orders_df["shipped_at"].notna()]

check(
    "shipped_at >= order_date",
    (shipped_orders["shipped_at"] >= shipped_orders["order_date"]).all()
)

delivered_orders_check = orders_df[orders_df["delivered_at"].notna()]

check(
    "delivered_at >= shipped_at",
    (delivered_orders_check["delivered_at"] >= delivered_orders_check["shipped_at"]).all()
)

payments_with_orders = payments_df.merge(
    orders_df[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

check(
    "payment_date >= order_date",
    (payments_with_orders["payment_date"] >= payments_with_orders["order_date"]).all()
)

check(
    "status_updated_at >= payment_date",
    (payments_df["status_updated_at"] >= payments_df["payment_date"]).all()
)


=== 6. COHERENCIA TEMPORAL ===

[OK] Ningún pedido anterior al registro del cliente
[OK] shipped_at >= order_date
[OK] delivered_at >= shipped_at
[OK] payment_date >= order_date
[OK] status_updated_at >= payment_date


In [114]:
review_check = (
    reviews_df
    .merge(
        order_items_df[["order_item_id", "order_id"]],
        on="order_item_id",
        how="left"
    )
    .merge(
        orders_df[["order_id", "status", "delivered_at"]],
        on="order_id",
        how="left"
    )
)

check(
    "Todas las reviews pertenecen a pedidos entregados",
    (review_check["status"] == "delivered").all()
)

check(
    "Todas las reviews son posteriores a la entrega",
    (review_check["review_date"] >= review_check["delivered_at"]).all()
)

[OK] Todas las reviews pertenecen a pedidos entregados
[OK] Todas las reviews son posteriores a la entrega


In [115]:
payment_check_items = order_items_df.copy()

# Importe real de cada línea
payment_check_items["calculated_line_total"] = payment_check_items.apply(
    lambda row: (Decimal(str(row["quantity"])) 
                 * row["unit_price"]
                 * (Decimal("1") - (row["discount_percent"] / Decimal("100")))
                ),
                axis=1
)

# Suma de las líneas de cada pedido
payment_check_totals = (
    payment_check_items.groupby("order_id")["calculated_line_total"].sum().reset_index()
)

# Inclusión de gastos de envío
payment_check_totals = (
    payment_check_totals
    .merge(
        orders_df[["order_id", "shipping_cost"]],
        on="order_id",
        how="left"
    )
)

# Cálculo de importe final esperado
payment_check_totals["expected_amount"] = payment_check_totals.apply(
    lambda row: money(row["calculated_line_total"] + row["shipping_cost"]),
    axis=1
)

# Inclusión del importe almacenado en payments
payment_check_totals = (
    payment_check_totals
    .merge(
        payments_df[["order_id", "amount"]],
        on="order_id",
        how="left"
    )
)

# Comparación de ambos importes
check(
    "Todos los importes de pago coinciden con pedido + envío",
    (payment_check_totals["expected_amount"] == payment_check_totals["amount"]).all()
)

[OK] Todos los importes de pago coinciden con pedido + envío


In [116]:
print("\n=== RESULTADO FINAL ===\n")

validation_df = pd.DataFrame(validation_results)

error_count = (validation_df["status"] == "ERROR").sum()

ok_count = (validation_df["status"] == "OK").sum()


print(f"Comprobaciones correctas: {ok_count}")

print(f"Errores encontrados: {error_count}")


if error_count == 0:
    print("\nVALIDACIÓN COMPLETADA SIN ERRORES")

else:
    print("\nREVISAR LAS COMPROBACIONES MARCADAS COMO ERROR")


=== RESULTADO FINAL ===

Comprobaciones correctas: 71
Errores encontrados: 0

VALIDACIÓN COMPLETADA SIN ERRORES


In [117]:
validation_df[validation_df["status"] == "ERROR"]

,check,status


## Carga de datos en BigQuery

A partir de aquí se cargan los datos generados a las tablas previamente creadas en BigQuery.  
Las tablas se cargan respetando las dependencias y posteriormente se verifican los registros.

In [118]:
from pathlib import Path
from dotenv import load_dotenv
from google.cloud import bigquery
import os


ROOT_DIR = Path.cwd()

if not (ROOT_DIR / ".env").exists():
    ROOT_DIR = ROOT_DIR.parent.parent


ENV_PATH = ROOT_DIR / ".env"

load_dotenv(ENV_PATH)


PROJECT_ID = os.getenv("GCP_PROJECT_ID")

DATASET_ID = os.getenv("BQ_DATASET_ID")


credentials_path = Path(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))

if not credentials_path.is_absolute():
    credentials_path = ROOT_DIR / credentials_path

credentials_path = credentials_path.resolve()

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path)


client = bigquery.Client(project=PROJECT_ID)

print(f"Conectado a BigQuery: {PROJECT_ID}.{DATASET_ID}")

Conectado a BigQuery: tc-sql-bometon.skeletia


In [119]:
# Orden de carga de las tablas

TABLE_LOAD_ORDER = [
    "countries",
    "cities",
    "acquisition_channels",
    "categories",
    "brands",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews",
]

In [120]:
# Primero vaciado de tablas antes de cargar la información
def clear_bigquery_tables() -> None:
    """
    Vacía las tablas de BigQuery antes de volver a cargar
    los datos sintéticos.

    Se recorren en orden inverso a sus dependencias.
    """

    for table_name in reversed(TABLE_LOAD_ORDER):

        table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

        sql = f"TRUNCATE TABLE `{table_id}`"

        client.query(sql).result()

        print(f"[OK] {table_name} vaciada")

In [121]:
# Carga de DataFrame
def load_dataframe_to_bigquery(
    table_name: str,
    dataframe: pd.DataFrame,
) -> bool:
    """
    Carga un DataFrame en una tabla existente de BigQuery.

    Devuelve True si la carga termina correctamente
    y False si se produce algún error.
    """

    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    try:

        # Recuperamos la tabla ya creada para utilizar
        # exactamente su esquema de BigQuery.
        table = client.get_table(table_id)

        job_config = bigquery.LoadJobConfig()

        job_config.schema = table.schema

        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND

        load_job = (
            client.load_table_from_dataframe(
                dataframe,
                table_id,
                job_config=job_config
            )
        )

        # Esperamos a que termine la carga.
        load_job.result()

        print(f"[OK] {table_name}: {len(dataframe)} filas cargadas")

        return True


    except Exception as error:
        print(f"[ERROR] {table_name}: {error}")
        return False

In [122]:
# Carga completa de las 11 tablas
def load_all_dataframes() -> bool:
    """
    Carga todos los DataFrames en BigQuery
    respetando el orden de dependencias.
    """

    all_ok = True

    for table_name in TABLE_LOAD_ORDER:

        dataframe = DATAFRAMES[table_name]

        success = load_dataframe_to_bigquery(table_name, dataframe)

        if not success:
            all_ok = False
            break

    return all_ok

In [123]:
# Ejecutamos la limpieza
clear_bigquery_tables()

[OK] reviews vaciada
[OK] payments vaciada
[OK] order_items vaciada
[OK] orders vaciada
[OK] products vaciada
[OK] customers vaciada
[OK] brands vaciada
[OK] categories vaciada
[OK] acquisition_channels vaciada
[OK] cities vaciada
[OK] countries vaciada


In [127]:
load_success = load_all_dataframes()

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] countries: 8 filas cargadas


/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] cities: 24 filas cargadas


/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


...

[OK] order_items: 4500 filas cargadas


/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] payments: 2000 filas cargadas


/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] reviews: 1031 filas cargadas


### Verificación de la carga a BigQuery

Comprobación que todas las tablas tienen los mismos registros que los DataFrame.

In [128]:
def verify_bigquery_load() -> pd.DataFrame:
    """
    Compara el número de filas de cada DataFrame
    con el número de filas almacenadas en BigQuery.
    """

    results = []

    for table_name in TABLE_LOAD_ORDER:

        table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

        # Número de filas que esperamos encontrar.
        expected_rows = len(DATAFRAMES[table_name])

        # Número real de filas almacenadas en BigQuery.
        query = f"""
        SELECT COUNT(*) AS row_count
        FROM `{table_id}`
        """

        query_result = client.query(query).result()

        actual_rows = next(query_result)["row_count"]

        # Comparamos ambos valores.
        status = "OK" if actual_rows == expected_rows else "ERROR"

        results.append(
            {
                "table": table_name,
                "dataframe_rows": expected_rows,
                "bigquery_rows": actual_rows,
                "status": status,
            }
        )

        print(f"[{status}] {table_name}: {actual_rows} / {expected_rows} filas")

    return pd.DataFrame(results)

In [129]:
load_verification = verify_bigquery_load()

[OK] countries: 8 / 8 filas
[OK] cities: 24 / 24 filas
[OK] acquisition_channels: 5 / 5 filas
[OK] categories: 8 / 8 filas
[OK] brands: 10 / 10 filas
[OK] customers: 500 / 500 filas
[OK] products: 70 / 70 filas
[OK] orders: 2000 / 2000 filas
[OK] order_items: 4500 / 4500 filas
[OK] payments: 2000 / 2000 filas
[OK] reviews: 1031 / 1031 filas


In [130]:
all_tables_loaded = (load_verification["status"] == "OK").all()


if all_tables_loaded:
    print("\nCARGA EN BIGQUERY VERIFICADA CORRECTAMENTE")

else:
    print("\nSE HAN DETECTADO ERRORES EN LA CARGA")


CARGA EN BIGQUERY VERIFICADA CORRECTAMENTE
